# Time-Phased Multi-Period Planning & Network Decomposition

Every solver used in `01`–`07` (`scripts/utils.py::build_and_solve_ttr`/`build_and_solve_tts`, plus the 8 objective variants) is a **single-snapshot** LP: one scalar horizon `t`, a disruption that's either on or off for the whole horizon, and no notion of a shipment departing now and arriving later. This notebook demonstrates two **new, additive** capabilities modeled after Adexa's two biggest structural advantages over that kind of model — real time buckets with lead-time-offset material flow, and network decomposition so a what-if doesn't require re-solving the entire network at full fidelity:

- **`scripts/multi_period_planning.py`** — a time-phased multi-period LP: discrete periods, per-node lead-time offsets (a shipment departing period `t` only arrives `offset[i]` periods later), and inventory that carries over from one period to the next.
- **`scripts/network_aggregation.py`** — aggregate/disaggregate decomposition: pool similar low-individual-impact tier-3 nodes into synthetic group nodes, solve the smaller aggregate network, then restore only the subset that actually needs full fidelity for a given disruption.

Neither module modifies `scripts/utils.py`, `scripts/disruption_scenarios.py`, `scripts/realistic_topologies.py`, `scripts/dataset_io.py`, or `scripts/company_profiles.py` — they sit alongside the existing engine and reuse it read-only where useful (`_prep_lp_data`, `derive_indexes`).

## Cluster Configuration
This notebook was tested on the following Databricks cluster configuration:
- **Databricks Runtime Version:** 17.3 LTS ML (includes Apache Spark 4.0.0, Scala 2.13)
- **Single Node**
    - Azure: Standard_DS4_v2 (28 GB Memory, 8 Cores)
    - AWS: m5d.2xlarge (32 GB Memory, 8 Cores)
- **Photon Acceleration:** Disabled (Photon boosts Apache Spark workloads; not all ML workloads will see an improvement)

No Ray needed — we stay at `medium` (~700-node) scale, matching `06`.

In [ ]:
%pip install -r ./requirements.txt --quiet
dbutils.library.restartPython()

In [ ]:
import json
import random

import matplotlib.pyplot as plt
import pandas as pd

import scripts.disruption_scenarios as ds_lib
import scripts.multi_period_planning as mpp
import scripts.network_aggregation as agg_lib
import scripts.scenario_calibration as sc

In [ ]:
catalog = "supply_chain_stress_test"  # Change here
schema = "data"                       # Change here
volume = "operational"                # Change here

## Load the Medium Dataset

Generated in `05_realistic_operational_data`. We add cost/holding/lead-time/emissions fields via `scenario_calibration.calibrate_cost_fields` — `production_delay` is what drives each supplier's lead-time offset below.

In [ ]:
with open(f"/Volumes/{catalog}/{schema}/{volume}/dataset_realistic_medium.json", "r") as f:
    medium = json.load(f)

rng = random.Random(1)
cost_fields = sc.calibrate_cost_fields(
    rng,
    medium["tier1"], medium["tier2"], medium["tier3"],
    medium["material_types"], medium["supplier_material_type"],
    medium["edges"], medium.get("criticality"), "medium",
    region=medium.get("region"),
)
medium = {**medium, **cost_fields}
print(f"tier1={len(medium['tier1'])}, tier2={len(medium['tier2'])}, tier3={len(medium['tier3'])}")

## Baseline: Flat vs. Seasonal Demand

`default_periodic_demand` flat-repeats each tier-1 product's demand across every period by default — total demand over the horizon matches what a single-snapshot `build_and_solve_ttr` run would see at `t = n_periods * period_length_days`, so the two engines are comparable at baseline. Passing a `seasonality` map (from `scenario_calibration.sample_seasonality_index`, opt-in) layers a per-node sinusoidal multiplier (mean 1.0) on top, so peak and trough periods no longer look identical.

In [ ]:
n_periods = 10
period_length_days = 7.0

d_t_flat = mpp.default_periodic_demand(medium, n_periods, period_length_days)

seasonality_rng = random.Random(2)
seasonality = sc.sample_seasonality_index(seasonality_rng, medium["tier1"], n_periods, "medium")
d_t_seasonal = mpp.default_periodic_demand(medium, n_periods, period_length_days, seasonality=seasonality)

sample_node = medium["tier1"][0]
pd.DataFrame({
    "flat": d_t_flat[sample_node],
    "seasonal": d_t_seasonal[sample_node],
}).plot(figsize=(8, 4), title=f"{sample_node}: flat vs. seasonal demand by period", marker="o")
plt.xlabel("period")
plt.ylabel("demand")
plt.tight_layout()
plt.show()

## Baseline Solve: No Disruption

We use the seasonal demand table from here on — it's a more realistic picture of period-to-period variation than a flat repeat. `build_and_solve_time_phased_ttr` returns the same `termination_condition`/`lost_profit` summary as `build_and_solve_ttr`, now scoped to the whole horizon; pass `return_model=True` to also get the Pyomo model for `extract_time_phased_solution`'s long-format per-(node, period) table.

In [ ]:
baseline = mpp.build_and_solve_time_phased_ttr(
    medium, [], n_periods, period_length_days, d_t=d_t_seasonal, return_model=True
)
print(baseline[["termination_condition", "lost_profit"]])
baseline_sol = mpp.extract_time_phased_solution(baseline.iloc[0]["model"], medium)

### Cold-Start Ramp: Lead-Time Offset in Action

`sample_node`'s upstream suppliers have lead times mostly in the 3–15 day range, i.e. `offset = ceil(lead_time_days / 7)` of 1–2 periods. With zero starting downstream inventory (`initial_pipeline` isn't supplied, so pre-horizon shipments default to zero — a documented cold-start simplification), the very first periods show a ramp: some lost demand while the first round-trip of shipments is still in transit, then recovery once material starts arriving on schedule.

In [ ]:
baseline_sol[baseline_sol["node"] == sample_node].set_index("period")[["produced", "inventory", "lost"]]

## Disruption: A Long-Lead-Time Supplier Goes Down Mid-Horizon

Unlike a scalar `ttr` in `DisruptionScenario`, a `TimePhasedDisruption` has a `period_start` and `duration_periods` — it begins at a specific period and lasts a specific number of periods, not "the whole horizon." We pick the node with the single longest lead time (the one a real recovery would be hardest to expedite around) and take it down for 4 periods starting at period 1.

In [ ]:
worst_node = max(medium["production_delay"], key=medium["production_delay"].get)
print(
    f"{worst_node}: lead_time_days={medium['production_delay'][worst_node]}, "
    f"criticality={medium['criticality'][worst_node]}, material={medium['supplier_material_type'][worst_node]}"
)

disruption = mpp.TimePhasedDisruption(
    scenario_id="demo_long_lead_time",
    name="Long-lead-time supplier down",
    description=f"{worst_node} offline for 4 periods starting period 1",
    real_world_basis="Illustrative — no real-world citation.",
    scenario_type="single_node",
    disrupted_nodes=[worst_node],
    period_start=1,
    duration_periods=4,
)

disrupted = mpp.build_and_solve_time_phased_ttr(
    medium, [disruption], n_periods, period_length_days, d_t=d_t_seasonal, return_model=True
)
print(disrupted[["termination_condition", "lost_profit"]])
disrupted_sol = mpp.extract_time_phased_solution(disrupted.iloc[0]["model"], medium)

### Per-Period Lost Profit: Baseline vs. Disrupted

Summing lost volume times margin across every tier-1 product per period shows exactly when the disruption bites — not just the total, aggregate figure a single-snapshot TTR run would give.

In [ ]:
def lost_profit_by_period(sol: pd.DataFrame, dataset: dict) -> pd.Series:
    v = sol[sol["node"].isin(dataset["tier1"])].copy()
    v["lost_profit"] = v["lost"] * v["node"].map(dataset["f"])
    return v.groupby("period")["lost_profit"].sum()

pd.DataFrame({
    "baseline": lost_profit_by_period(baseline_sol, medium),
    "disrupted": lost_profit_by_period(disrupted_sol, medium),
}).plot(figsize=(8, 4), title="Lost profit by period: baseline vs. disrupted", marker="o")
plt.xlabel("period")
plt.ylabel("lost profit")
plt.tight_layout()
plt.show()

### Inventory Trajectory: The Disrupted Node Itself

`worst_node`'s own inventory should hold flat at 0 through its 4 disrupted periods (production forced to 0, nothing to carry over) and recover once the disruption window ends.

In [ ]:
disrupted_sol[disrupted_sol["node"] == worst_node].set_index("period")[["produced", "inventory"]].plot(
    figsize=(8, 4), title=f"{worst_node}: production and inventory across the disruption window", marker="o"
)
plt.axvspan(disruption.period_start, disruption.period_start + disruption.duration_periods - 1, alpha=0.15, color="red")
plt.xlabel("period")
plt.tight_layout()
plt.show()

## Aggregate/Disaggregate Decomposition

`build_aggregate_dataset` pools tier-3 nodes sharing the same material and criticality tag into synthetic group nodes (`c_agg = Σc_i`, `s_agg = Σs_i`) — monopoly-bottleneck nodes are never pooled, since an aggregate node can't be "35% disrupted" and pooling away the network's actual single points of failure would defeat the point of this whole accelerator. `run_aggregate_then_disaggregate` orchestrates the full workflow: solve the smaller aggregate network, compute which real nodes need to be restored to full fidelity (`disaggregation_scope` — the disrupted node's entire group, its downstream consumer cone, and its upstream alternate suppliers), rebuild a mixed-fidelity dataset, and re-solve just that.

In [ ]:
outcome = agg_lib.run_aggregate_then_disaggregate(
    medium, [disruption], n_periods=n_periods, period_length_days=period_length_days, d_t=d_t_seasonal,
)

print("node counts:", outcome["node_counts"])
print("timings (s):", {k: round(v, 3) for k, v in outcome["timings"].items()})
print("\nfull network lost_profit:     ", disrupted.iloc[0]["lost_profit"])
print("aggregate network lost_profit:", outcome["aggregate_result"].iloc[0]["lost_profit"])
print("partial (disaggregated) lost_profit:", outcome["partial_result"].iloc[0]["lost_profit"])

The partial (disaggregated) solve reproduces the full network's `lost_profit` while using a structurally smaller model (`partial_tier3` < `full_tier3` above) — the disrupted node's own group, its downstream consumers, and its upstream alternate suppliers get full fidelity; everything else stays rolled up. The pure-aggregate solve is close but not guaranteed to match exactly: aggregation error is **one-directional and optimistic** (pooled capacity/inventory and unioned edges only ever loosen constraints relative to the true network), which is exactly why disaggregation restores full fidelity around the disrupted node rather than trusting the aggregate number there.

Wall-clock timings above are illustrative for this notebook's ~700-node scale, not a hard performance claim — the relative benefit of decomposition grows with network size, since the aggregate/partial solve's variable count grows with `partial_tier3`, not `full_tier3`.

## Wrap Up

This notebook demonstrated the two additive capabilities modeled after Adexa's structural advantages over a single-snapshot LP: time-phased multi-period planning (discrete periods, lead-time-offset shipments, inventory carryover, a disruption with a real start/duration instead of an all-or-nothing scalar `ttr`) and aggregate/disaggregate network decomposition (solve a smaller pooled network, then restore only the subset that needs full fidelity for a given disruption). Both sit alongside — and never modify — the existing `build_and_solve_ttr`/`build_and_solve_tts` engine and disruption-scenario library used throughout `01`–`07`. See the README's "Time-Phased Multi-Period Planning & Network Decomposition" section for the full schema, worked examples, and documented limitations (cold-start pipeline, end-of-horizon effect, MIP scale growth, fixed-radius disaggregation scope, one-directional optimistic aggregation error).

&copy; 2025 Databricks, Inc. All rights reserved. The source in this notebook is provided subject to the Databricks License [https://databricks.com/db-license-source].  All included or referenced third party libraries are subject to the licenses set forth below.

| library                                | description             | license    | source                                              |
|----------------------------------------|-------------------------|------------|-----------------------------------------------------|
| pyomo | An object-oriented algebraic modeling language in Python for structured optimization problems | BSD-3 | https://pypi.org/project/pyomo/
| highspy | Linear optimization solver (HiGHS) | MIT | https://pypi.org/project/highspy/